# Lab 07 — Connected Component Labeling (Two-Pass Algorithm)

**Course:** Image Analysis and Processing — École des Mines de Saint-Étienne  
**Topics covered:**
1. Two-pass 8-connectivity Connected Component Labeling (CCL) implemented from scratch
2. Label equivalence table construction and transitive merging
3. Visual benchmark against `skimage.measure.label`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import filters, measure

## 1. Two-Pass Connected Component Labeling Implementation

In raster-scan order (top-to-bottom, left-to-right), the 8-connected causal neighborhood of pixel $(i, j)$ consists of 4 previously visited neighbors: top, left, top-left, and top-right.
- **Pass 1:** Assign provisional labels and record label equivalences whenever distinct neighboring labels meet.
- **Equivalence Resolution:** Merge transitively connected equivalence sets.
- **Pass 2:** Relabel every foreground pixel with the canonical representative of its equivalence class.

In [ ]:
def check_neighbors(label_img: np.ndarray, i: int, j: int) -> list:
    """Return non-zero labels among the 4 causal 8-connected neighbors of (i, j)."""
    neighbors = []
    _, cols = label_img.shape
    if i > 0 and label_img[i - 1, j] != 0:
        neighbors.append(label_img[i - 1, j])
    if j > 0 and label_img[i, j - 1] != 0:
        neighbors.append(label_img[i, j - 1])
    if i > 0 and j > 0 and label_img[i - 1, j - 1] != 0:
        neighbors.append(label_img[i - 1, j - 1])
    if i > 0 and j < cols - 1 and label_img[i - 1, j + 1] != 0:
        neighbors.append(label_img[i - 1, j + 1])
    return neighbors


def merge_equivalence_table(equivalence_table: dict) -> dict:
    """Merge transitively linked label equivalence sets."""
    keys = list(equivalence_table.keys())
    for key in keys:
        for other_key, other_values in list(equivalence_table.items()):
            if key != other_key and key in other_values and key in equivalence_table:
                equivalence_table[other_key].update(equivalence_table[key])
                equivalence_table.pop(key)
                break
    return equivalence_table


def connected_components_first_pass(binary_image: np.ndarray):
    """First pass: provisional labeling and equivalence conflict recording."""
    equivalences = {}
    next_label = 0
    labeled = np.zeros(binary_image.shape, dtype=int)

    for i in range(binary_image.shape[0]):
        for j in range(binary_image.shape[1]):
            if binary_image[i, j] > 0:
                neighbors = check_neighbors(labeled, i, j)
                if not neighbors:
                    next_label += 1
                    labeled[i, j] = next_label
                else:
                    min_label = min(neighbors)
                    labeled[i, j] = min_label
                    for nb_label in neighbors:
                        if nb_label != min_label:
                            equivalences.setdefault(min_label, set()).add(nb_label)

    return labeled, merge_equivalence_table(equivalences)


def relabel_second_pass(labeled_image: np.ndarray, equivalences: dict) -> np.ndarray:
    """Second pass: replace provisional labels with canonical labels."""
    output = labeled_image.copy()
    lookup = {}
    for canonical, aliases in equivalences.items():
        for alias in aliases:
            lookup[alias] = canonical

    for i in range(output.shape[0]):
        for j in range(output.shape[1]):
            if output[i, j] in lookup:
                output[i, j] = lookup[output[i, j]]
    return output

## 2. Benchmark Against `skimage.measure.label`

We generate synthetic binary blobs via Gaussian-smoothed random impulses and compare our from-scratch implementation against scikit-image.

In [ ]:
n = 12
size = 256
rng = np.random.default_rng(1)
im = np.zeros((size, size))

points = (size * rng.random((2, n**2))).astype(int)
im[points[0], points[1]] = 1.0
im = filters.gaussian(im, sigma=size / (4.0 * n))
blobs = im > 0.7 * im.mean()

all_labels = measure.label(blobs)
blobs_labels = measure.label(blobs, background=0)

provisional, eq_table = connected_components_first_pass(blobs)
custom_labels = relabel_second_pass(provisional, eq_table)

n_skimage = len(np.unique(blobs_labels)) - 1
n_custom = len(np.unique(custom_labels)) - 1
print(f"Components found — scikit-image: {n_skimage} | Custom two-pass: {n_custom}")

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
panels = [
    (blobs, "Binary Blobs", "gray"),
    (all_labels, "skimage (all regions)", "nipy_spectral"),
    (blobs_labels, "skimage (background=0)", "nipy_spectral"),
    (custom_labels, "Custom Two-Pass CCL", "nipy_spectral"),
]
for ax, (img, title, cmap) in zip(axes, panels):
    ax.imshow(img, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()